# Stage 7: the figures and tables of the paper

**Reads:** `data/gravity_clean.csv`, saved by stage 2; `results/cleaning_log.csv` (stage 2); `results/dev_runs.csv` and `results/dev_components.csv` (stage 3); `results/size_rule.csv`, `results/gaps.csv` and `results/cutoff.csv` (stage 4); `results/analysis_runs.csv`, `results/analysis_summary.csv`, `results/analysis_cutoff.csv` and `results/selection_confusion.csv` (stage 5); `results/check_runs.csv`, `results/check_components.csv` and `results/check_cutoff.csv` (stage 6).

**Writes:** into `figures/`, every figure and table of the paper (Requirement 4.11), listed in `figures/README.md`:
- eight tables, each as `.csv` with the full numbers and as `.tex`, the same table rounded for the paper: `tab_cleaning`, `tab_summary`, `tab_tuned`, `tab_cutoff`, `tab_selection`, `tab_selection_confusion`, `tab_checks` and `tab_trees2x`;
- five figures, each as `.pdf`, with a `.csv` of the numbers it draws: `learning_curves`, `components`, `gaps`, `analysis_methods` and `noise_gaps`.

Stage 7 of section 6 of `docs/build_plan.md`. Nothing is fitted here and no random number is drawn: every number comes from the files of stages 2 to 6, and the functions that turn them into tables and figures are in `helpers.py`. Before anything is written, the files are checked against each other: the tuned settings, gaps and patterns are computed again from the draws with the functions of stage 4 and must equal those in the files. The `.tex` files hold only a LaTeX `tabular` with the rules of the `booktabs` package; captions are Yu's and go in the paper, and so does any note a table needs.

The development year (2017) is for exploring: its ensembles are tuned on its own test set, so its results are exploratory, and so are those of the checks of stage 6 run on it. The analysis year (2019) is read at the settings frozen in decision 21, and its results test H1 and H2. Every standard error and interval here describes variation over training draws from one pool, scored on one test set. The draws overlap, more so at large sizes.

*Drafted with Claude (Anthropic), which was asked for stage 7 of the build plan after Yu's run of stage 6: the figures and tables of the paper, made from the files of stages 2 to 6. It replaces the placeholder of 18 October 2026.*

In [ ]:
from helpers import (ANALYSIS_YEAR, CANDIDATES, CURVE_LINES, CV_DRAWS, DATA, DEV_YEAR, DEVIANCE_LABEL, DRAWS,
                     FIGURES, GAP_LABEL, RESULTS, ROOT, SIZE_LABEL, SIZES, TEXT_COLOUR, analysis_summary,
                     axis_limits, check_find_crossing, checks_table, checks_text, cleaning_table, cleaning_text,
                     confusion_overall, confusion_text, crossing_marker, crossings, cutoff_overview, cutoff_text,
                     draw_line, experiment_gaps, gap_shares, largest_relative_difference, latex_tabular,
                     learning_curves, line_configs, refuse_part, selection_confusion_table, selection_table,
                     selection_text, size_axis, summary_statistics, summary_text, trees2x_table, trees2x_text,
                     tuned_table, tuned_text, with_reversal)

import io

import matplotlib.pyplot as plt
import pandas as pd

refuse_part()
# Section 8: find_crossing on toy gaps, before the patterns are computed again.
check_find_crossing()

# Text sizes for figures drawn at the width of a page (6.4 inches).
plt.rcParams.update({"axes.labelsize": 9, "axes.titlesize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
                     "legend.fontsize": 8})
FIGURES.mkdir(exist_ok=True)


def save_table(numbers, text_and_align, name):
    # The full numbers as .csv; the text of the paper as .tex, and printed here.
    text, align = text_and_align
    numbers.to_csv(FIGURES / f"{name}.csv", index=False, lineterminator="\n")
    (FIGURES / f"{name}.tex").write_text(latex_tabular(text, align))
    with pd.option_context("display.width", 250, "display.max_columns", None, "display.max_colwidth", 60):
        print(text.to_string(index=False))


def save_figure(figure, numbers, name):
    # The figure as .pdf, with no creation date in the file, so that the same
    # files give the same figure; the numbers it draws as .csv beside it.
    figure.savefig(FIGURES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None})
    numbers.to_csv(FIGURES / f"{name}.csv", index=False, lineterminator="\n")
    plt.show()


def print_beyond(beyond):
    # The values a figure draws at the edge of its axis, with their true value.
    if beyond:
        print("Drawn at the edge of the axis, with the value written beside them:")
        print(pd.DataFrame(beyond).to_string(index=False, float_format="{:.4g}".format))
    else:
        print("Every value is inside the axis.")


print(f"Development year {DEV_YEAR}, analysis year {ANALYSIS_YEAR}; sizes {list(SIZES)}; {DRAWS} draws per size")
print(f"Writing into {FIGURES.relative_to(ROOT)}/")

## The files of stages 2 to 6, checked against each other

The files must be those of the full runs: every size of `SIZES` with `DRAWS` draws in the development year (the base run and the three checks) and in the analysis year. Then the numbers that later cells read from one file and draw from another must agree:

- `experiment_gaps` (stage 4 steps 1, 2 and 4, the function stage 6 uses) on `dev_runs.csv` and `dev_components.csv` must give the tuned settings of `size_rule.csv` and the gaps of `gaps.csv`, and `crossings` on those gaps the patterns of `cutoff.csv`;
- the same on the files of `noise` and `split2` must give the patterns of `check_cutoff.csv`;
- `analysis_summary` and `selection_confusion_table` (stage 5) on `analysis_runs.csv` must give `analysis_summary.csv` and `selection_confusion.csv`.

Each comparison is to a relative 1e-9, apart from the seconds. The bootstrap intervals are read from the files and not computed again.

In [ ]:
def read(name):
    # round_trip reads every number back exactly as the stage wrote it.
    return pd.read_csv(RESULTS / name, float_precision="round_trip")


def as_file(table):
    # The table as it reads back from a .csv file, so that its columns have the types of a file the stage wrote.
    return pd.read_csv(io.StringIO(table.to_csv(index=False)), float_precision="round_trip")


def agrees(new, old, keys, what):
    # Every row of old must be in new, and every shared number equal to a relative 1e-9.
    difference, shared = largest_relative_difference(new, old, keys)
    assert shared == len(old) == len(new), f"{what}: {shared} rows shared, out of {len(new)} and {len(old)}."
    assert difference <= 1e-9, f"{what}: the files differ by {difference:.1e}. Stop and report."
    print(f"{what}: {shared} rows, largest relative difference {difference:.1e}")


log = read("cleaning_log.csv")
runs_base, components_base = read("dev_runs.csv"), read("dev_components.csv")
rule, gaps_file, cutoff_base = read("size_rule.csv"), read("gaps.csv"), read("cutoff.csv")
analysis_runs, summary_2019 = read("analysis_runs.csv"), read("analysis_summary.csv")
cutoff_2019, confusion = read("analysis_cutoff.csv"), read("selection_confusion.csv")
check_runs, check_components, check_cutoff = read("check_runs.csv"), read("check_components.csv"), read("check_cutoff.csv")

# Full runs only: every size, and DRAWS draws of every configuration or method at each size (CV_DRAWS for
# cross-validation, decision 4).
for name, table, key in [("dev_runs.csv", runs_base, "config"), ("analysis_runs.csv", analysis_runs, "method")]:
    assert sorted(table["size"].unique()) == list(SIZES), f"{name} does not have every size of SIZES."
    draws = table.groupby(["size", key])["draw"].nunique().reset_index()
    expected = [CV_DRAWS if value == "cv_select" else DRAWS for value in draws[key]]
    assert (draws["draw"] == expected).all(), f"{name} does not have the draws of a full run."
assert (runs_base["variant"] == "base").all(), "dev_runs.csv holds rows other than the base run."
for run in ["noise", "split2"]:
    of_run = check_runs[check_runs["variant"] == run]
    assert sorted(of_run["size"].unique()) == list(SIZES), f"{run} does not have every size of SIZES."
    assert (of_run.groupby(["size", "config"])["draw"].nunique() == DRAWS).all(), f"{run} lacks draws."

# The base run of the development year.
summary_base, tuned_base, gaps_base = experiment_gaps(runs_base, components_base)
tuned_file = rule[rule["model"].isin(["rf", "gbm"])][["size", "model", "config"]]
agrees(tuned_base[["size", "model", "config"]], tuned_file, ["size", "model"], "Tuned rf and gbm against size_rule.csv")
agrees(gaps_base, gaps_file, ["size", "comparison"], "Gaps against gaps.csv")
crossing_columns = ["comparison", "pattern", "n_star", "last_crossing", "crossing_from", "crossing_to"]
agrees(crossings(gaps_base)[crossing_columns], cutoff_base[crossing_columns], ["comparison"],
       "Patterns against cutoff.csv")

# The checks noise and split2, each with rf and gbm tuned on its own test set, as in stage 6.
check_gaps = {}
for run in ["noise", "split2"]:
    _, _, check_gaps[run] = experiment_gaps(check_runs[check_runs["variant"] == run],
                                            check_components[check_components["variant"] == run])
    agrees(crossings(check_gaps[run])[crossing_columns],
           check_cutoff[check_cutoff["variant"] == run][crossing_columns], ["comparison"],
           f"Patterns of {run} against check_cutoff.csv")

# The analysis year.
agrees(as_file(analysis_summary(analysis_runs)), summary_2019, ["size", "method"], "analysis_summary.csv from analysis_runs.csv")
agrees(as_file(selection_confusion_table(analysis_runs)), confusion, ["selector", "sizes", "picked"],
       "selection_confusion.csv from analysis_runs.csv")

## `tab_cleaning`: the rows after each cleaning step

From `cleaning_log.csv`, which stage 2 wrote: the rows of each year in the extract, then the rows dropped and the rows left after each step, with the reason. The rows left after the last step must be the rows of that year in `gravity_clean.csv`.

In [ ]:
# Country identifiers are text, as in stage 2: no value is read as missing.
clean = pd.read_csv(DATA / "gravity_clean.csv", dtype={"country_id_o": str, "country_id_d": str, "pair_id": str},
                    keep_default_na=False, na_values=[""])
cleaning = cleaning_table(log, clean)
save_table(cleaning, cleaning_text(cleaning), "tab_cleaning")

## `tab_summary`: the cleaned table

For the target, `tradeflow_baci` in thousand current USD, and for each of the eight predictors, in each year: the rows, mean, standard deviation, minimum, median and maximum, over every usable row of `gravity_clean.csv` (the test sets included). The mean of an indicator is the share of rows where it is 1.

In [ ]:
statistics = summary_statistics(clean)
save_table(statistics, summary_text(statistics), "tab_summary")

## `tab_tuned`: the tuned settings and errors of the development year

`size_rule.csv`, frozen in decision 21: at each size, `ppml`, tuned `rf` and tuned `gbm` with the mean test deviance over the draws, its standard error and the median, and the model the size rule picks. "One draw over half" marks a mean that one draw holds more than half of, so that its standard error is unreliable. The last column says which tuned setting sits at an edge of its grid, as the checkpoint defines an edge (section 6): "possible" where the grid cannot go further, "in the grid" where it could have but the one round of the rule Grid edges was used at the checkpoint (decision 19). The test errors are those the ensembles were tuned on, so they favour the ensembles.

In [ ]:
tuned = tuned_table(rule)
save_table(tuned, tuned_text(tuned), "tab_tuned")

## `tab_cutoff`: the pattern of the error gap and the cutoff

One row per comparison and run, read with decision 12. Three runs of the development year, all exploratory: the base run (`cutoff.csv`), and the checks `noise` and `split2` (`check_cutoff.csv`), each with `rf` and `gbm` tuned on its own test set. Then the analysis year at the frozen settings (`analysis_cutoff.csv`): the row "ppml vs better ensemble" is the test of H1, and the row "ridge vs better ensemble" the cutoff test of `ridge` beside it (decision 22), which is not part of H1.

"Reversal" gives the two sizes around the change from `ppml` behind to `ppml` ahead, which decision 12 asks the paper to report. `cutoff.csv` and `check_cutoff.csv` do not hold them, so they are found here on the error gaps with `reversal_sizes`, the function of stage 5. "Crossing" is n* on `one_crossing` and the last crossing on `reversal_then_crossing`, with the 2.5 and 97.5 percentiles over the bootstrap resamples that have the same pattern, and the number of those resamples. The last six columns are the share of the resamples in each pattern.

In [ ]:
cutoffs = cutoff_overview([
    (DEV_YEAR, "base", with_reversal(cutoff_base, gaps_base)),
    (DEV_YEAR, "noise", with_reversal(check_cutoff[check_cutoff["variant"] == "noise"], check_gaps["noise"])),
    (DEV_YEAR, "split2", with_reversal(check_cutoff[check_cutoff["variant"] == "split2"], check_gaps["split2"])),
    (ANALYSIS_YEAR, "analysis", cutoff_2019),
])
save_table(cutoffs, cutoff_text(cutoffs), "tab_cutoff")

## `tab_selection`: the size rule against cross-validation in the analysis year (H2)

From the rows `size_rule` and `cv_select` of `analysis_summary.csv`, at each size: the model the size rule picks, the mean test deviance of each selector with its standard error in brackets, and the paired difference size rule minus cross-validation over the draws that have both, with its standard error. "Within 2 SE" is the test of H2 at that size: the difference is at most twice its standard error. Then the paired difference size rule minus `best_of_three`, which looks at the test set and so is a reference, not a selector; and the fits and seconds per draw of each selector. The fits do not depend on the machine; the seconds are those of the run in the Nuvolos workspace.

In [ ]:
selection = selection_table(summary_2019)
save_table(selection, selection_text(selection), "tab_selection")

## `tab_selection_confusion`: the model picked against the model that won

The rows of `selection_confusion.csv` over all sizes together, for each selector: the model it picked against the model with the lowest test deviance of the three on the same draw (the winner `best_of_three` found). On each row `total`: the share of draws picked right, the share that always picking the most frequent winner of each size would get (no selector that picks one model per size can do better), and the mean difference to `best_of_three` with its paired standard error. The share picked right is descriptive: H2 is read on the deviance alone.

In [ ]:
overall = confusion_overall(confusion)
save_table(overall, confusion_text(overall), "tab_selection_confusion")

## `tab_checks` and `tab_trees2x`: the checks of stage 6

`tab_checks`: `ppml` against the better ensemble at each size in the base run and in the checks `noise` and `split2`, all in the development year: the better ensemble, the error gap E(ppml) - E(c) with its paired standard error, the draws in which `ppml` has the lower test deviance, and the bias gap and variance gap in the version `rn` (section 7), which stages 4 and 6 print. In each row, error gap = bias gap - variance gap. The patterns these gaps give are in `tab_cutoff`. The two splits have different test sets, so the levels of `split2` are measured on other test pairs.

`tab_trees2x`: the check `trees2x` at the largest size, on the training samples of the base run: each configuration with more trees against the tuned one, with the mean test deviance of both, the mean of the paired difference (negative: more trees lowered the error) with its standard error, and the draws in which more trees gave the lower test deviance.

In [ ]:
checks = checks_table({"base": gaps_base, "noise": check_gaps["noise"], "split2": check_gaps["split2"]})
save_table(checks, checks_text(checks), "tab_checks")
print()
largest = max(SIZES)
trees2x = trees2x_table(check_runs[check_runs["variant"] == "trees2x"], runs_base, rule[rule["size"] == largest])
save_table(trees2x, trees2x_text(trees2x), "tab_trees2x")

## The figures

Every figure draws training size on a log axis, with a tick at each size. The value axis of each panel is set by the values at every size but the smallest, with some room around them. At the smallest size the mean test deviance of `ppml` can rest on one draw, as the column `se_unreliable` of `size_rule.csv` and `analysis_summary.csv` shows, and drawn to scale it would flatten every other line. A value outside the axis is drawn at its edge as a triangle pointing beyond it, with the value written beside it, and each cell below prints those values. Each figure's numbers are written beside it as a `.csv`.

### `learning_curves`: test and training deviance in the development year

For `ppml`, tuned `rf`, tuned `gbm`, the reference lines `mean` and `tree`, and the reference line `ridge`, which Yu added at the checkpoint (decision 20): the mean test deviance over the draws (solid) with a band of two standard errors, and the mean training deviance (dashed), defined as in item 1 of the checkpoint. The training deviance is in-sample for `ppml`, `gbm`, `mean` and `ridge`, out-of-bag for `rf`, and absent for `tree`, and it follows the tuned setting at each size. Both axes are logarithmic. The functions are those of the checkpoint (`line_configs`, `learning_curves`), with the ridge line added.

In [ ]:
CURVE_LABELS = {"ppml": "ppml", "rf": "rf, tuned", "gbm": "gbm, tuned", "mean": "mean (reference)",
                "tree": "tree (reference)", "ridge": "ridge (reference, added at the checkpoint)"}
curves = learning_curves(summary_base, line_configs(tuned_base, CURVE_LINES))
sizes = sorted(int(n) for n in curves["size"].unique())
larger = curves[curves["size"] > min(sizes)]
limits = axis_limits(pd.concat([larger["mean"], larger["lower"], larger["upper"], larger["train_deviance"]]), log=True)

figure, axes = plt.subplots(figsize=(6.4, 4.6), layout="constrained")
axes.set_yscale("log")
written, beyond = {}, []
for line in CURVE_LINES:
    rows = curves[curves["line"] == line].sort_values("size")
    for mark in draw_line(axes, rows["size"], rows["mean"], line, CURVE_LABELS[line], limits, written,
                          band=(rows["lower"], rows["upper"])):
        beyond.append({"line": line, "deviance": "test", **mark})
    if rows["train_deviance"].notna().any():
        for mark in draw_line(axes, rows["size"], rows["train_deviance"], line, None, limits, written, style="--",
                              name=f"{line}, training"):
            beyond.append({"line": line, "deviance": "training", **mark})
# Two grey lines that only explain, in the legend, what solid and dashed mean.
axes.plot([], [], color=TEXT_COLOUR, linestyle="-", label="test deviance, two standard errors shaded")
axes.plot([], [], color=TEXT_COLOUR, linestyle="--", label="training deviance (out-of-bag for rf)")
axes.set_ylim(limits)
size_axis(axes, sizes)
axes.set_xlabel(SIZE_LABEL)
axes.set_ylabel(DEVIANCE_LABEL)
axes.grid(True, which="major", color="#e5e5e2", linewidth=0.8)
figure.legend(frameon=False, ncol=2, loc="outside lower center")
save_figure(figure, curves, "learning_curves")
print_beyond(beyond)

### `components`: variance and remainder of the three models in the development year

For `ppml`, tuned `rf` and tuned `gbm` at each size, the two parts of the mean test deviance of section 7: the variance V, how much the predictions of one draw differ from their mean over draws, and the remainder B, the rest of the error (bias and noise), in the version `rn`. The two add up to the mean test deviance. The version `rn` corrects the variance for the number of draws and for the overlap of samples drawn from one pool, so it can exceed the error, and B is then negative; a negative value cannot be drawn on a log axis and is drawn at the bottom edge with its value.

In [ ]:
COMPONENT_LABELS = {"ppml": "ppml", "rf": "rf, tuned", "gbm": "gbm, tuned"}
lines = line_configs(tuned_base)
parts = components_base.merge(lines[lines["line"].isin(CANDIDATES)], on=["size", "config"])
parts = parts[["size", "line", "config", "draws", "error", "variance_rn", "remainder_rn"]].sort_values(["line", "size"])
larger = parts[parts["size"] > min(sizes)]
limits = axis_limits(pd.concat([larger["variance_rn"], larger["remainder_rn"]]), log=True)

figure, panels = plt.subplots(1, 2, figsize=(6.4, 3.3), sharey=True, layout="constrained")
beyond = []
for axes, column, title in zip(panels, ["variance_rn", "remainder_rn"], ["Variance V", "Remainder B (bias and noise)"]):
    axes.set_yscale("log")
    written = {}
    for line in CANDIDATES:
        rows = parts[parts["line"] == line]
        for mark in draw_line(axes, rows["size"], rows[column], line, COMPONENT_LABELS[line], limits, written):
            beyond.append({"line": line, "part": column, **mark})
    axes.set_ylim(limits)
    size_axis(axes, sizes)
    axes.set_title(title)
    axes.set_xlabel(SIZE_LABEL)
    axes.grid(True, which="major", color="#e5e5e2", linewidth=0.8)
panels[0].set_ylabel(DEVIANCE_LABEL)
panels[0].legend(frameon=False)
save_figure(figure, parts, "components")
print_beyond(beyond)

### `gaps`: the bias gap and the variance gap in the development year, with the cutoff

One panel for each comparison of stage 4: `ppml` against tuned `rf`, against tuned `gbm`, and against the better of the two. Each draws the error gap E(ppml) - E(c), the bias gap B(ppml) - B(c) and the variance gap V(c) - V(ppml), in the version `rn`, so that error gap = bias gap - variance gap. Below zero, `ppml` has the lower error (error gap), the lower remainder (bias gap), or the larger variance (variance gap). Each gap is divided by the mean test deviance of the ensemble at that size (`gap_shares`), because every error falls as the training size grows and a gap in thousand USD at 5,000 pairs would be too small to read beside one at 100. The signs, and so the patterns and n*, do not change. The dotted vertical line is n* or the last crossing of `cutoff.csv`, with its bootstrap interval shaded; a pattern without a crossing has no line.

In [ ]:
GAP_LINES = [("error_gap", "error_gap_share", "error gap"), ("bias_gap", "bias_gap_rn_share", "bias gap"),
             ("variance_gap", "variance_gap_rn_share", "variance gap")]
PANEL_TITLES = {"rf": "ppml against tuned rf", "gbm": "ppml against tuned gbm",
                "best": "ppml against the better of the two"}


def draw_gaps(axes, gap_table, limits, written, style, label_suffix):
    # The three gaps of one run in one panel; returns the values drawn at an edge.
    marks = []
    rows = gap_table.sort_values("size")
    for key, column, label in GAP_LINES:
        for mark in draw_line(axes, rows["size"], rows[column], key, label + label_suffix, limits, written,
                              style=style):
            marks.append({"gap": label, **mark})
    return marks


def draw_crossing(axes, cutoff_row, style, text_row, prefix):
    # n* or the last crossing as a vertical line, with its bootstrap interval shaded and its size written at the
    # top; text_row moves the text down one row, so that two runs' texts do not overlap.
    marker = crossing_marker(cutoff_row)
    if marker is None:
        return
    size, low, high, name = marker
    if pd.notna(low) and pd.notna(high):
        axes.axvspan(low, high, color="#d9d9d6", alpha=0.5, linewidth=0, zorder=0)
    axes.axvline(size, color=TEXT_COLOUR, linestyle=style, linewidth=1.2)
    axes.annotate(f"{prefix}{name} {size:,.0f}", (size, 1), xycoords=("data", "axes fraction"),
                  xytext=(-4, -12 - 10 * text_row), textcoords="offset points", ha="right", fontsize=7,
                  color=TEXT_COLOUR)


shares_base = gap_shares(gaps_base)
larger = shares_base[shares_base["size"] > min(sizes)]
limits = axis_limits(larger[[column for _, column, _ in GAP_LINES]].to_numpy().ravel(), log=False)

figure, panels = plt.subplots(3, 1, figsize=(6.4, 7.8), sharex=True, layout="constrained")
beyond = []
for axes, comparison in zip(panels, ["rf", "gbm", "best"]):
    written = {}
    axes.axhline(0, color="#b4b2a9", linewidth=0.8)
    for mark in draw_gaps(axes, shares_base[shares_base["comparison"] == comparison], limits, written, "-", ""):
        beyond.append({"comparison": comparison, **mark})
    draw_crossing(axes, cutoff_base[cutoff_base["comparison"] == comparison].iloc[0], ":", 0, "")
    axes.set_ylim(limits)
    size_axis(axes, sizes)
    axes.set_title(PANEL_TITLES[comparison])
    axes.grid(True, which="major", color="#e5e5e2", linewidth=0.8)
panels[-1].set_xlabel(SIZE_LABEL)
figure.supylabel(GAP_LABEL, fontsize=9)
handles, labels = panels[0].get_legend_handles_labels()
figure.legend(handles, labels, frameon=False, ncol=3, loc="outside lower center")
save_figure(figure, shares_base, "gaps")
print_beyond(beyond)

### `analysis_methods`: the analysis year at the frozen settings

From `analysis_summary.csv`: the mean test deviance over the draws of each method, with a band of two standard errors, in the analysis year. Left, the three models at the settings frozen in decision 21. Right, the two selectors, the size rule and cross-validation, and `best_of_three`, the lowest test deviance of the three on each draw, which looks at the test set and so is a reference, not a selector. The size rule's line is the line of the model it picks at that size. Both axes are logarithmic.

In [ ]:
METHOD_LABELS = {"ppml": "ppml", "rf": "rf, frozen setting", "gbm": "gbm, frozen setting", "size_rule": "size rule",
                 "cv_select": "cross-validation", "best_of_three": "best of three (looks at the test set)"}
PANELS = [("The three models", ["ppml", "rf", "gbm"]), ("Selecting a model", ["size_rule", "cv_select", "best_of_three"])]
methods = summary_2019[summary_2019["method"].isin(list(METHOD_LABELS))][["size", "method", "model", "config", "draws",
                                                                  "mean", "se"]].copy()
methods["lower"] = methods["mean"] - 2 * methods["se"]
methods["upper"] = methods["mean"] + 2 * methods["se"]
larger = methods[methods["size"] > min(sizes)]
limits = axis_limits(pd.concat([larger["mean"], larger["lower"], larger["upper"]]), log=True)

figure, panels = plt.subplots(1, 2, figsize=(6.4, 3.4), sharey=True, layout="constrained")
beyond = []
for axes, (title, of_panel) in zip(panels, PANELS):
    axes.set_yscale("log")
    written = {}
    for method in of_panel:
        rows = methods[methods["method"] == method].sort_values("size")
        for mark in draw_line(axes, rows["size"], rows["mean"], method, METHOD_LABELS[method], limits, written,
                              band=(rows["lower"], rows["upper"])):
            beyond.append({"method": method, **mark})
    axes.set_ylim(limits)
    size_axis(axes, sizes)
    axes.set_title(title)
    axes.set_xlabel(SIZE_LABEL)
    axes.grid(True, which="major", color="#e5e5e2", linewidth=0.8)
    axes.legend(frameon=False)
panels[0].set_ylabel(DEVIANCE_LABEL)
save_figure(figure, methods.sort_values(["method", "size"]), "analysis_methods")
print_beyond(beyond)

### `noise_gaps`: the gaps with and without noise in the training flows

The panels of `gaps`, for the base run (solid) and the check `noise` of stage 6 (dashed), in which every training flow was multiplied by a random factor of mean 1 and variance `NOISE_VAR` (decision 5). Each run's gaps are divided by the mean test deviance of its own ensemble, and each run has its own tuned `rf` and `gbm`. The vertical lines are n* or the last crossing of each run, dotted for the base run and dash-dotted for `noise`, with their bootstrap intervals shaded.

In [ ]:
shares_noise = gap_shares(check_gaps["noise"])
both = pd.concat([shares_base.assign(run="base"), shares_noise.assign(run="noise")], ignore_index=True)
larger = both[both["size"] > min(sizes)]
limits = axis_limits(larger[[column for _, column, _ in GAP_LINES]].to_numpy().ravel(), log=False)
cutoff_noise = check_cutoff[check_cutoff["variant"] == "noise"]

RUNS = [("base", shares_base, "-", ":", cutoff_base), ("noise", shares_noise, "--", "-.", cutoff_noise)]
figure, panels = plt.subplots(3, 1, figsize=(6.4, 8.2), sharex=True, layout="constrained")
beyond = []
for axes, comparison in zip(panels, ["rf", "gbm", "best"]):
    written = {}
    axes.axhline(0, color="#b4b2a9", linewidth=0.8)
    for text_row, (run, of_run, style, crossing_style, cutoff) in enumerate(RUNS):
        for mark in draw_gaps(axes, of_run[of_run["comparison"] == comparison], limits, written, style, f", {run}"):
            beyond.append({"run": run, "comparison": comparison, **mark})
        draw_crossing(axes, cutoff[cutoff["comparison"] == comparison].iloc[0], crossing_style, text_row, f"{run}: ")
    axes.set_ylim(limits)
    size_axis(axes, sizes)
    axes.set_title(PANEL_TITLES[comparison])
    axes.grid(True, which="major", color="#e5e5e2", linewidth=0.8)
panels[-1].set_xlabel(SIZE_LABEL)
figure.supylabel(GAP_LABEL, fontsize=9)
handles, labels = panels[0].get_legend_handles_labels()
figure.legend(handles, labels, frameon=False, ncol=2, loc="outside lower center")
save_figure(figure, both[["run"] + list(shares_base.columns)], "noise_gaps")
print_beyond(beyond)

## What was written

In [ ]:
for path in sorted(FIGURES.iterdir()):
    if path.suffix in (".csv", ".tex", ".pdf"):
        print(f"{path.relative_to(ROOT)}: {path.stat().st_size:,} bytes")